# Data 3 — location, grid strain, and cost

The question for this table is: **given a latitude and longitude, what does the California grid look like there, and what does electricity cost?**

One row is one day at one coordinate. The test file covers the three years from **2023-10-19 through 2026-10-18**, at five California points (San Diego, Los Angeles, Fresno, Sacramento, San Francisco). A full daily file would be about 5,500 rows. This file keeps every fifth complete day, the same dates at every city, which lands near **1,000 rows**. Five cities are enough to see that weather changes with the coordinate. The grid number for a day does not.

Two targets:

- **`grid_strain_mwh`** is the day's CAISO demand that solar, wind, hydro, nuclear, and geothermal did not cover. It is the energy left for fossil plants, storage, and imports. A high number is a strained day. A low or negative number means the zero-carbon fleet covered the load.
- **`cost_cents_per_kwh`** is California's residential retail price for that month, from EIA. It is the cost series we actually have. It does not change inside the state, and it does not change inside the month. Wholesale hourly prices are not in this pull.

Weather is the part that belongs to the coordinate. Strain and price are California-wide, joined on the date. For a new point, fetch that point's weather and reuse the same day's grid columns. This file only stores the five test cities.

Sources, all fetched when this notebook runs (nothing in the repo is a pre-built table):

- EIA daily generation by fuel and daily demand for CAISO (`CISO`), Pacific dates.
- EIA monthly California residential retail price.
- Open-Meteo daily weather at the five coordinates. No key.

EIA reads `EIA_API_KEY` from `.env`. The saved file is `data/processed/location_strain_cost.csv`.

## What gets dropped

### Circular: the column is the target, or it is an ingredient of the target

Strain is computed as demand minus solar, wind, hydro, nuclear, and geothermal. After that subtraction, those inputs are deleted. A model that saw them would be redoing the subtraction, not predicting it.

| Dropped | Why it is circular |
| --- | --- |
| Same-day fuel megawatt-hours (`NG`, `COL`, `OIL`, `SUN`, `WND`, `WAT`, `NUC`, `GEO`, other, storage) | Solar, wind, hydro, nuclear, and geothermal are the terms that were subtracted. Gas, coal, and oil are how that leftover energy was actually served. |
| Fuel shares, total generation, carbon intensity | Shares and carbon are the fuel mix rewritten as a ratio. Carbon is not a second measurement of strain. |
| `demand_mwh` | It is the first term of the strain formula. |
| Net generation | The generation side of the same daily balance as demand. |
| Interchange, including each neighboring balancing authority | Demand, generation, and imports are one equation. Neighbor columns are that equation split into pieces. |
| Retail `revenue` | Revenue is price times sales. |
| Same-month `sales` | Monthly sales are the slow version of demand. |

Past strain is not circular. `strain_lag_7d_mwh` and `strain_lag_14d_mwh` are earlier days. Last month's price is not this month's price, so `cost_lag_1m_cents` stays.

### Dumb: a real column that repeats another column or does not belong on this row

| Dropped | Why it is not usable here |
| --- | --- |
| Sunshine duration | The same sunlight signal as radiation. |
| Cooling and heating degree days | A rewrite of temperature. Temperature is already on the row. |
| The other four cities' weather | This row has a coordinate. Weather from a different coordinate belongs on that coordinate's row. |
| Customer counts, use per customer, other sectors, transportation | Not strain, and not the residential price. |
| Commercial and industrial prices | A second price, not a property of the coordinate. |
| Open-Meteo forecast runs and the short CAISO solar forecast | They cover days or weeks, not these three years, so they would be empty on almost every row. |
| Hour-of-day columns | This file is daily. |
| Raw day-of-year | 31 December and 1 January would look far apart. The sine and cosine pair keeps the year circular. |

## What stays

| Column | Role |
| --- | --- |
| `date`, `location_name` | Row labels. The name is the city we fetched. Do not use it as a predictor next to the coordinates; it is the same fact twice. |
| `latitude`, `longitude` | The input the question is about. |
| `year`, `month`, `dow` | Calendar. Price moves with the year. Load moves with the month and the weekday. |
| `doy_sin`, `doy_cos` | Season, with December next to January. |
| `temp_mean_c`, `temp_max_c`, `temp_min_c` | Temperature at that coordinate. Max is the hot afternoon, min is the night. |
| `precip_mm`, `wind_max_kmh`, `radiation_mj` | Rain, the day's strongest surface wind, and sunlight at that coordinate. |
| `demand_forecast_mwh` | CAISO's day-ahead demand forecast. It is known before the day, and it is not the demand value inside the strain formula. |
| `strain_lag_7d_mwh`, `strain_lag_14d_mwh` | Strain a week ago and two weeks ago. |
| `cost_lag_1m_cents` | Residential price in the previous month. |
| `grid_strain_mwh` | Target. Same value at every coordinate on a given day. |
| `cost_cents_per_kwh` | Target. Same value everywhere in California that month. Blank from August 2026 on, because EIA had not published those months yet. Strain and weather are still filled on those rows. |

In [ ]:
import json
import os
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from dotenv import load_dotenv


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / ".env.example").exists():
            return candidate
    return here


ROOT = find_repo_root()
load_dotenv(ROOT / ".env")

FETCH_START = "2023-09-01"
SAMPLE_START = pd.Timestamp("2023-10-19")
SAMPLE_END = pd.Timestamp("2026-10-18")
TARGET_ROWS = 1000
CLEAN_FUELS = ["SUN", "WND", "WAT", "NUC", "GEO"]
LOCATIONS = [
    ("san_diego", 32.7157, -117.1611),
    ("los_angeles", 34.0522, -118.2437),
    ("fresno", 36.7378, -119.7871),
    ("sacramento", 38.5816, -121.4944),
    ("san_francisco", 37.7749, -122.4194),
]
WEATHER_VARS = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "shortwave_radiation_sum",
]
KEEP_COLUMNS = [
    "date",
    "location_name",
    "latitude",
    "longitude",
    "year",
    "month",
    "dow",
    "doy_sin",
    "doy_cos",
    "temp_mean_c",
    "temp_max_c",
    "temp_min_c",
    "precip_mm",
    "wind_max_kmh",
    "radiation_mj",
    "demand_forecast_mwh",
    "strain_lag_7d_mwh",
    "strain_lag_14d_mwh",
    "cost_lag_1m_cents",
    "grid_strain_mwh",
    "cost_cents_per_kwh",
]
OUT_PATH = ROOT / "data" / "processed" / "location_strain_cost.csv"


def get_json(url: str, timeout: int = 180) -> dict:
    request = Request(url, headers={"User-Agent": "EcoShift-Data3/1.0"})
    payload = None
    for attempt in range(4):
        try:
            with urlopen(request, timeout=timeout) as response:
                payload = json.load(response)
            break
        except HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            if exc.code in {429, 500, 502, 503} and attempt < 3:
                time.sleep(65 if exc.code == 429 else 5 * (attempt + 1))
                continue
            route = exc.url.split("?", 1)[0]
            raise RuntimeError(f"Request failed ({exc.code}) for {route}: {detail[:400]}") from None
    if not isinstance(payload, dict):
        raise RuntimeError("empty response")
    if payload.get("error") and "response" not in payload and "daily" not in payload:
        raise RuntimeError(str(payload.get("reason") or payload.get("error")))
    return payload


def eia_pages(route: str, params: list) -> pd.DataFrame:
    api_key = os.getenv("EIA_API_KEY")
    if not api_key:
        raise RuntimeError("EIA_API_KEY is missing from .env")
    rows: list[dict] = []
    offset = 0
    total = None
    while True:
        query = [("api_key", api_key), ("offset", str(offset)), ("length", "5000"), *params]
        payload = get_json("https://api.eia.gov/v2/" + route + "?" + urlencode(query))
        response = payload["response"]
        batch = response.get("data") or []
        total = int(response["total"])
        rows.extend(batch)
        offset += len(batch)
        if not batch or offset >= total:
            break
        time.sleep(0.15)
    frame = pd.DataFrame(rows)
    if "value" in frame.columns:
        frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    print(f"{route}: {len(frame)} rows (api total {total})")
    return frame

In [ ]:
fuel = eia_pages(
    "electricity/rto/daily-fuel-type-data/data/",
    [
        ("data[]", "value"),
        ("facets[respondent][]", "CISO"),
        ("facets[timezone][]", "Pacific"),
        ("start", FETCH_START),
        ("end", SAMPLE_END.strftime("%Y-%m-%d")),
        ("sort[0][column]", "period"),
        ("sort[0][direction]", "asc"),
    ],
)
region = eia_pages(
    "electricity/rto/daily-region-data/data/",
    [
        ("data[]", "value"),
        ("facets[respondent][]", "CISO"),
        ("facets[timezone][]", "Pacific"),
        ("start", FETCH_START),
        ("end", SAMPLE_END.strftime("%Y-%m-%d")),
        ("sort[0][column]", "period"),
        ("sort[0][direction]", "asc"),
    ],
)
retail = eia_pages(
    "electricity/retail-sales/data/",
    [
        ("data[]", "price"),
        ("frequency", "monthly"),
        ("facets[stateid][]", "CA"),
        ("start", "2023-08"),
        ("end", SAMPLE_END.strftime("%Y-%m")),
        ("sort[0][column]", "period"),
        ("sort[0][direction]", "asc"),
    ],
)

weather_frames = []
for name, latitude, longitude in LOCATIONS:
    payload = get_json(
        "https://archive-api.open-meteo.com/v1/archive?"
        + urlencode(
            [
                ("latitude", str(latitude)),
                ("longitude", str(longitude)),
                ("start_date", FETCH_START),
                ("end_date", SAMPLE_END.strftime("%Y-%m-%d")),
                ("daily", ",".join(WEATHER_VARS)),
                ("timezone", "America/Los_Angeles"),
            ]
        )
    )
    frame = pd.DataFrame(payload["daily"])
    frame["location_name"] = name
    frame["latitude"] = latitude
    frame["longitude"] = longitude
    weather_frames.append(frame)
    print(f"weather {name}: {len(frame)} days")
    time.sleep(0.4)
weather = pd.concat(weather_frames, ignore_index=True)

In [ ]:
fuels = fuel.pivot_table(index="period", columns="fueltype", values="value", aggfunc="sum")
fuels.index = pd.to_datetime(fuels.index).tz_localize(None).normalize()
for column in CLEAN_FUELS:
    if column not in fuels.columns:
        fuels[column] = 0.0
    fuels[column] = pd.to_numeric(fuels[column], errors="coerce").fillna(0.0).clip(lower=0.0)

region_wide = region.pivot_table(index="period", columns="type", values="value", aggfunc="sum")
region_wide.index = pd.to_datetime(region_wide.index).tz_localize(None).normalize()
if "D" not in region_wide.columns or "DF" not in region_wide.columns:
    raise RuntimeError(f"EIA region data is missing demand columns: {list(region_wide.columns)}")

grid = pd.DataFrame(index=fuels.index.union(region_wide.index).sort_values())
grid["demand_mwh"] = region_wide["D"]
grid["demand_forecast_mwh"] = region_wide["DF"]
grid["clean_mwh"] = fuels[CLEAN_FUELS].sum(axis=1)
grid["grid_strain_mwh"] = grid["demand_mwh"] - grid["clean_mwh"]
grid = grid.reindex(pd.date_range(grid.index.min(), grid.index.max(), freq="D"))
grid["strain_lag_7d_mwh"] = grid["grid_strain_mwh"].shift(7)
grid["strain_lag_14d_mwh"] = grid["grid_strain_mwh"].shift(14)
grid = grid.loc[(grid.index >= SAMPLE_START) & (grid.index <= SAMPLE_END)].copy()
grid = grid.dropna(subset=["grid_strain_mwh", "demand_forecast_mwh", "strain_lag_7d_mwh", "strain_lag_14d_mwh"])

price = retail.loc[retail["sectorid"] == "RES", ["period", "price"]].copy()
price["price"] = pd.to_numeric(price["price"], errors="coerce")
price = price.dropna(subset=["price"]).drop_duplicates("period")
price["month_period"] = pd.PeriodIndex(price["period"], freq="M")
price = price.set_index("month_period").sort_index()
price = price.reindex(pd.period_range(price.index.min(), price.index.max(), freq="M"))
price["cost_cents_per_kwh"] = price["price"]
price["cost_lag_1m_cents"] = price["cost_cents_per_kwh"].shift(1)

weather["date"] = pd.to_datetime(weather["time"]).dt.tz_localize(None).dt.normalize()
weather = weather.rename(
    columns={
        "temperature_2m_mean": "temp_mean_c",
        "temperature_2m_max": "temp_max_c",
        "temperature_2m_min": "temp_min_c",
        "precipitation_sum": "precip_mm",
        "wind_speed_10m_max": "wind_max_kmh",
        "shortwave_radiation_sum": "radiation_mj",
    }
)

grid_frame = grid.drop(columns=["demand_mwh", "clean_mwh"]).reset_index(names="date")
panel = weather.merge(grid_frame, on="date", how="inner")
panel["month_period"] = pd.PeriodIndex(panel["date"], freq="M")
panel = panel.merge(
    price[["cost_cents_per_kwh", "cost_lag_1m_cents"]],
    left_on="month_period",
    right_index=True,
    how="left",
)

local = panel["date"].dt.tz_localize("America/Los_Angeles")
panel["year"] = local.dt.year.astype(int)
panel["month"] = local.dt.month.astype(int)
panel["dow"] = local.dt.dayofweek.astype(int)
day_of_year = local.dt.dayofyear.to_numpy()
panel["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
panel["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
panel["date"] = panel["date"].dt.strftime("%Y-%m-%d")

needed = [column for column in KEEP_COLUMNS if column not in {"cost_cents_per_kwh", "cost_lag_1m_cents"}]
panel = panel.dropna(subset=needed)
counts = panel.groupby("date")["location_name"].nunique()
complete_dates = counts[counts == len(LOCATIONS)].index.sort_values()
step = max(1, round(len(complete_dates) * len(LOCATIONS) / TARGET_ROWS))
kept_dates = complete_dates[::step]
sample = panel[panel["date"].isin(kept_dates)].loc[:, KEEP_COLUMNS].sort_values(
    ["date", "location_name"]
).reset_index(drop=True)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(OUT_PATH, index=False)
print(f"complete days {len(complete_dates)}, kept every {step} days -> {len(kept_dates)} days")
print(f"wrote {OUT_PATH}  shape {sample.shape}")
print(f"{sample['date'].min()} to {sample['date'].max()}")
print(f"cost missing on {int(sample['cost_cents_per_kwh'].isna().sum())} rows")
sample.head(10)

In [ ]:
banned = {
    "NG", "COL", "OIL", "SUN", "WND", "WAT", "NUC", "GEO", "OTH",
    "demand_mwh", "clean_mwh", "net_generation_mwh", "interchange_mwh",
    "carbon_intensity_g_per_kwh", "gas_share", "solar_share", "wind_share",
    "generation_mwh", "sales", "revenue", "customers", "sunshine_duration",
}
present = banned & set(sample.columns)
if present:
    raise RuntimeError(f"circular or dropped columns are still in the file: {sorted(present)}")

one_day = sample[sample["date"] == sample["date"].iloc[0]]
if one_day["grid_strain_mwh"].nunique() != 1:
    raise RuntimeError("strain changed inside a single day; the grid join duplicated dates")
if one_day["temp_max_c"].nunique() < 2:
    raise RuntimeError("temperature did not change across coordinates")
if not (800 <= len(sample) <= 1300):
    raise RuntimeError(f"expected about 1000 rows, got {len(sample)}")

print("locations:", ", ".join(sorted(sample["location_name"].unique())))
print("columns:", ", ".join(sample.columns))
print(sample.groupby("location_name")[["latitude", "longitude"]].first())